In [1]:
from langchain.chat_models import init_chat_model
from topic_gen.generate import Generator
from topic_gen.models import MTO_responds
from langchain_community.llms import VLLM

from topic_gen import logger
logger.setLevel("INFO")

In [2]:
import torch
import os
print(f"Cuda: {torch.cuda.is_available()}")
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
print(f"Number of GPUs: {torch.cuda.device_count()}")

Cuda: True
Number of GPUs: 1


In [3]:
import ir_datasets

dataset = ir_datasets.load("disks45/nocr/trec-robust-2004")
store = dataset.docs_store()

queries_map = {}
for q in dataset.queries_iter():
    queries_map[q.query_id] = q
    
qrels_map = {}
for q in dataset.qrels_iter():
    if q.query_id not in qrels_map:
        qrels_map[q.query_id] = []
    qrels_map[q.query_id].append(q)

In [4]:
model_dir = "../data/raw/datasets/cache/"
model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"

llm = VLLM(model=model_name, 
          download_dir=model_dir,
          # trust_remote_code=True,
          temperature=0.7,
          vllm_kwargs={
            "max_model_len": 32768, 
            "max_num_batched_tokens": 32768,
            }
)

INFO 08-22 15:25:01 [__init__.py:241] Automatically detected platform cuda.
INFO 08-22 15:25:02 [utils.py:326] non-default args: {'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', 'download_dir': '../data/raw/datasets/cache/', 'max_model_len': 32768, 'max_num_batched_tokens': 32768, 'disable_log_stats': True}
INFO 08-22 15:25:09 [__init__.py:711] Resolved architecture: Qwen3MoeForCausalLM
INFO 08-22 15:25:09 [__init__.py:1750] Using max model len 32768
INFO 08-22 15:25:10 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 08-22 15:25:14 [__init__.py:241] Automatically detected platform cuda.
(EngineCore_0 pid=26444) INFO 08-22 15:25:16 [core.py:636] Waiting for init message from front-end.
(EngineCore_0 pid=26444) INFO 08-22 15:25:16 [core.py:74] Initializing a V1 LLM engine (v0.10.1.1) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', skip_tokenizer_init=False, tokeniz

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:02<00:06,  2.23s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:02<00:02,  1.26s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:05<00:01,  1.73s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:07<00:00,  2.05s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:07<00:00,  1.91s/it]
(EngineCore_0 pid=26444) 


(EngineCore_0 pid=26444) INFO 08-22 15:25:26 [default_loader.py:262] Loading weights took 8.03 seconds
(EngineCore_0 pid=26444) WARNING 08-22 15:25:26 [marlin_utils_fp8.py:80] Your GPU does not have native support for FP8 computation but FP8 quantization is being used. Weight-only FP8 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(EngineCore_0 pid=26444) INFO 08-22 15:25:28 [gpu_model_runner.py:2007] Model loading took 29.5422 GiB and 10.309083 seconds
(EngineCore_0 pid=26444) INFO 08-22 15:25:37 [backends.py:548] Using cache directory: /home/vscode/.cache/vllm/torch_compile_cache/631635ac22/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_0 pid=26444) INFO 08-22 15:25:37 [backends.py:559] Dynamo bytecode transform time: 9.39 s
(EngineCore_0 pid=26444) INFO 08-22 15:25:46 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 8.260 s
(EngineCore_0 pid=26444) INFO 08-22 15:25:4

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:07<00:00,  8.43it/s]


(EngineCore_0 pid=26444) INFO 08-22 15:26:00 [gpu_model_runner.py:2708] Graph capturing finished in 9 secs, took 1.09 GiB
(EngineCore_0 pid=26444) INFO 08-22 15:26:00 [core.py:214] init engine (profile, create kv cache, warmup model) took 32.63 seconds
INFO 08-22 15:26:01 [llm.py:298] Supported_tasks: ['generate']


In [5]:
# llm = init_chat_model(
#     model="gemini-2.5-flash",
#     model_provider="google_genai",
#     temperature=0,
# )

In [6]:
generator = Generator(llm=llm, output_class=MTO_responds, prompt_name="robust-DNA-zero-shot")

[topic_gen] [INFO] Generating using prompt from '/home/vscode/.cache/pypoetry/virtualenvs/src-B2WAz2j2-py3.11/lib/python3.11/site-packages/topic_gen/prompts/robust-DNA-zero-shot.yaml'


In [7]:
query = queries_map["652"]

In [18]:
# prepare data
documents = []
queries = []
narratives = []
descriptions = []
for qrel in qrels_map[query.query_id]:
    document = store.get(qrel.doc_id).default_text()

    documents.append(document[:10000])
    queries.append(query.title)
    narratives.append(query.narrative)
    descriptions.append(query.description)

In [19]:
from ir_measures import Qrel

In [20]:
my_qrels = []
res = generator.generate(
    document=documents[:50], 
    query=queries[:50],
    narrative=narratives[:50],
    description=descriptions[:50],
    config={"max_concurrency": 1}
)


[topic_gen] [INFO] Generating batch of 50 items...
[topic_gen] [INFO] Final text prompt:
 Given a query and a web page, you must provide a score on an integer scale of 0 to 2 with the following meanings: 2 = highly relevant, very helpful for this query 1 = relevant, may be partly helpful but might contain other irrelevant content 0 = not relevant, should never be shown for this query
Assume that you are writing a report on the subject of the topic. If you would use any of the information contained in the web page in such a report, mark it 1. If the web page is primarily about the topic, or contains vital information about the topic, mark it 2. Otherwise, mark it 0.
Query A person has typed OIC Balkans 1990s into a search engine.
They were looking for: What was the OIC's involvement in the Balkans in 1990-94? Relevant documents describe the role the OIC played in the
Balkan region.  Also relevant are documents reflecting
the Balkans' players (nations, groups) positions pro or
con regard

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [22]:
res

[MTO_responds(M=1, T=1, O=0),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=0),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=0, O=0),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=0),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=1, T=1, O=1),
 MTO_responds(M=0, T=0, O=0),
 MTO_responds(M=1, T=1, O=1),
 MTO_respo